In [1]:
import os
		
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

d:\miniconda3\envs\PyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [4]:
ds = ds.train_test_split(test_size=0.2)
ds

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 5372
    })
})

In [5]:
ds['train'][:2]

{'output': ['您希望我为您做什么呢？您想让我回复您的电子邮件，还是给您设置闹钟，或者完成您的演示文稿呢？请给我更多信息，以便我更好地完成您的请求。',
  '这个示例是一种算法。\n\n解释：\n该示例是一种清晰的、定义明确、确定性的指导方针，适用于所有上周工作超过75小时的员工。这一决策基于确切的信息，即上周工作小时数。算法是一种确定性的过程，描述根据输入和一系列明确定义的步骤生成输出的方法。'],
 'input': ['', '输入：\n一家公司实施了一项规定，如果员工在上周工作超过75个小时，则可以休息一天。'],
 'instruction': ['回复电子邮件，设置闹钟，完成演示文稿。', '将以下示例分类为算法或启发式。']}

In [6]:
tokenizer = AutoTokenizer.from_pretrained("E:\modelscope\models\LLM-Research\Llama-3___2-1B")
tokenizer

PreTrainedTokenizerFast(name_or_path='E:\modelscope\models\LLM-Research\Llama-3___2-1B', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|reserved_

In [10]:
tokenizer.eos_token_id, tokenizer.eos_token

(128001, '<|end_of_text|>')

In [7]:
def process_func(example):
    MAX_LENGTH = 1024    # Llama分词器会将一个中文字切分为多个token，因此需要放开一些最大长度，保证数据的完整性
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ", add_special_tokens=False)
    response = tokenizer(example["output"], add_special_tokens=False)
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.eos_token_id]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [8]:
tokenized_ds = ds.map(process_func, remove_columns=ds['train'].column_names)
tokenized_ds

Map: 100%|██████████| 5372/5372 [00:01<00:00, 4231.14 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5372
    })
})

In [11]:
print(tokenized_ds['train'][0]["input_ids"])

[35075, 25, 115228, 59464, 111080, 65573, 14558, 3922, 45018, 13357, 117, 76161, 3922, 61648, 102446, 20379, 17161, 83932, 3490, 72803, 25, 220, 88126, 110526, 37046, 18184, 88126, 102210, 101879, 104586, 11571, 88126, 101067, 126997, 18904, 59464, 127442, 111080, 65573, 14558, 3922, 106302, 90112, 88126, 45018, 13357, 117, 76161, 3922, 108966, 61648, 127442, 102446, 20379, 17161, 83932, 104586, 11571, 15225, 124211, 117724, 28469, 105610, 102924, 37046, 34226, 53901, 30590, 61648, 127442, 35959, 1811, 128001]


In [12]:
tokenizer.decode(tokenized_ds['train'][0]["input_ids"])

'Human: 回复电子邮件，设置闹钟，完成演示文稿。\n\nAssistant: 您希望我为您做什么呢？您想让我回复您的电子邮件，还是给您设置闹钟，或者完成您的演示文稿呢？请给我更多信息，以便我更好地完成您的请求。<|end_of_text|>'

In [15]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds['train'][0]["labels"])))

'您希望我为您做什么呢？您想让我回复您的电子邮件，还是给您设置闹钟，或者完成您的演示文稿呢？请给我更多信息，以便我更好地完成您的请求。<|end_of_text|>'

In [26]:
tokenizer.pad_token = tokenizer.eos_token

In [17]:
import torch
model = AutoModelForCausalLM.from_pretrained("E:\modelscope\models\LLM-Research\Llama-3___2-1B", low_cpu_mem_usage=True, dtype=torch.bfloat16, device_map="auto")
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [18]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(task_type=TaskType.CAUSAL_LM,)
config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules=None, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)

In [19]:
model = get_peft_model(model, config)

In [20]:
model.enable_input_require_grads() 

In [21]:
model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [29]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    eval_strategy='steps',
    logging_steps=50,
    num_train_epochs=1,
)

In [30]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds['train'].select(range(6000)),
    eval_dataset=tokenized_ds['test'].select(range(1000)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

C:\Users\10433\AppData\Local\Temp\ipykernel_30544\1016864095.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [31]:
trainer.train()

Step,Training Loss,Validation Loss
50,2.070300,2.090806
100,2.028900,1.976417
150,1.954600,1.938461
200,1.889500,1.921984
250,1.910100,1.911770
300,1.906800,1.905948
350,1.885800,1.903409


TrainOutput(global_step=375, training_loss=1.9432487182617189, metrics={'train_runtime': 525.094, 'train_samples_per_second': 11.427, 'train_steps_per_second': 0.714, 'total_flos': 3287693507112960.0, 'train_loss': 1.9432487182617189, 'epoch': 1.0})

In [32]:
model.eval()
ipt = tokenizer("Human: {}\n{}".format("你好", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=512, do_sample=True, eos_token_id=tokenizer.eos_token_id)[0], skip_special_tokens=True)

'Human: 你好\n\nAssistant: 你好。'

In [33]:
ipt = tokenizer("Human: {}\n{}".format("生活要过的幸福关键是", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=512, do_sample=True, eos_token_id=tokenizer.eos_token_id)[0], skip_special_tokens=True)

'Human: 生活要过的幸福关键是\n\nAssistant: 生活要过的幸福关键是\n\nA. 自己\n\nB. 生活\n\nC. 自己和生活\n\nD. 自己和人\n\nE. 自己和幸福'

In [35]:
ipt = tokenizer("生活要过的幸福关键是", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=512, do_sample=True, eos_token_id=tokenizer.eos_token_id)[0], skip_special_tokens=True)

'生活要过的幸福关键是\nA. 自我实现\nB. 自我成熟\nC. 自我独立\nD. 自我安全\nE. 自我实现和自我成熟\nAnswer: E'